# Notebook 05: ICRA Research Gaps & Paper Ideas

**Goal:** Identify concrete, publishable research directions for an ICRA paper combining JEPA + VLA.

ICRA 2026 deadline is likely **Sep 2026**. You need a clear contribution that:
1. Builds on V-JEPA 2 / VLA-JEPA / JEPA-VLA
2. Has a novel, defensible contribution
3. Can be implemented with limited compute (AMD Radeon 8GB + cloud bursts)
4. Has clear benchmarks to evaluate against

---

## Landscape Map: What Exists vs What's Missing

```
                        ┌───────────────────────────────────────────┐
                        │          JEPA FOR ROBOTICS LANDSCAPE      │
                        └───────────────────────────────────────────┘

  DONE (by Meta):                    DONE (by others):           OPEN GAPS:
  ──────────────                     ──────────────────          ──────────
  ✅ V-JEPA 2 pretraining            ✅ JEPA-VLA (plug-in)      ❌ JEPA + Safety (CBF)
  ✅ V-JEPA 2-AC world model         ✅ VLA-JEPA (end-to-end)   ❌ Humanoid JEPA-VLA
  ✅ CEM planning in latent space    ✅ pi0 flow matching       ❌ Real-time JEPA planning
  ✅ Zero-shot manipulation          ✅ OpenVLA baseline         ❌ Multi-task world model
  ✅ DROID dataset training           ✅ CBF-RL for humanoids    ❌ JEPA + CBF latent safety
                                                                 ❌ Adaptive horizon planning
                                                                 ❌ JEPA for bimanual tasks
                                                                 ❌ Plug-in + World model combo
```

## 0.5 The 5W+H of JEPA for ICRA Research

### WHO publishes at ICRA?
**IEEE International Conference on Robotics and Automation** — the top robotics venue.
- Acceptance rate: ~43% (2024)
- Double-blind review, 6-8 pages
- Strong emphasis on: real experiments, ablation studies, clear contribution
- Typical timeline: submission Sep 2026, notification Jan 2027, conference May 2027

### WHAT makes a strong ICRA paper in this space?
1. **Clear, narrow contribution** — don't try to solve everything
2. **Reproducible results** — open code, standard benchmarks (LIBERO, DROID)
3. **Thorough ablations** — show each component matters
4. **Comparison to strong baselines** — JEPA-VLA, VLA-JEPA, OpenVLA-OFT
5. **Some real-robot results** — even limited, demonstrates feasibility

### WHERE does your work fit in the field?

```
                    High novelty, high risk
                         ▲
    Safe JEPA-VLA (CBF)  │  JEPA for Humanoids
                         │
          ─────────── Medium ───────────
                         │
    Unified JEPA-VLA  ★  │  Amortized Planning
    (YOUR BEST BET)      │
                         │
                    Low novelty, low risk
```

### WHEN should each phase happen?

| Month | Phase | Deliverable |
|-------|-------|-------------|
| **1-2** | Reproduce baselines | Working VLA-JEPA code on LIBERO |
| **3-4** | Build unified model | Gated cross-attention + world model working |
| **5-6** | Experiments | Full LIBERO results + ablations |
| **7** | Write paper | Submit draft |
| **8** | Revise + submit | Camera-ready for ICRA |

### WHY "Unified JEPA-VLA" as the recommended paper?

**Risk-reward analysis:**

| Paper Idea | Novelty | Risk | Compute | Timeline |
|------------|---------|------|---------|----------|
| Unified JEPA-VLA | Medium-High | **Low** | Medium | 4-5 months |
| Safe JEPA-VLA (CBF) | High | Medium-High | Low | 5-6 months |
| Amortized Planning | Medium | Medium | High | 5-6 months |
| Sim-to-Real | Medium | High | Medium | 6+ months |

Unified JEPA-VLA has:
- **Lowest risk:** Both components (features + world model) proven individually
- **Clearest contribution:** "First to combine both approaches"
- **Strong experimental design:** Head-to-head comparison on LIBERO
- **Available code:** Build on VLA-JEPA (open source)

### HOW does the Unified JEPA-VLA combine both approaches?

$$\mathcal{L}_{\text{unified}} = \underbrace{\mathcal{L}_{\text{FM}}(a_t, z_a)}_{\text{flow matching}} + \beta \underbrace{\mathcal{L}_{\text{WM}}(\hat{s}, s)}_{\text{world model}} + \gamma \underbrace{\mathcal{L}_{\text{feat}}(\text{VLM}, F(I))}_{\text{JEPA feature alignment}}$$

where:
- $\mathcal{L}_{\text{FM}}$: Flow matching loss for action prediction (from VLA-JEPA)
- $\mathcal{L}_{\text{WM}}$: World model prediction loss in JEPA latent space (from VLA-JEPA)
- $\mathcal{L}_{\text{feat}}$: Auxiliary loss encouraging VLM to attend to V-JEPA 2 features (from JEPA-VLA)
- $\beta = 0.1$, $\gamma$ = hyperparameter to tune

## Paper Idea 1: Safe JEPA-VLA (JEPA + CBF in Latent Space)

### Motivation
V-JEPA 2-AC can predict future states in latent space. CBFs define safe sets.
**Nobody has defined CBFs in JEPA's latent space.**

### Approach
```
Standard VLA:  obs + lang → VLA → action (no safety guarantee)

Safe JEPA-VLA:
  obs + lang → VLA → proposed action
                         ↓
  V-JEPA 2-AC: predict next latent state ŝ_{t+1} given proposed action
                         ↓
  Learned CBF h(ŝ_{t+1}): is predicted next state safe?
                         ↓
  If h(ŝ_{t+1}) > 0: execute action (safe)
  If h(ŝ_{t+1}) ≤ 0: solve QP to find closest safe action
```

### Key Equations

**Latent CBF constraint:**
$$h_\psi(P_\phi(a_t, s_t, z_t)) \geq -\alpha \cdot h_\psi(z_t)$$

where $h_\psi$ is a neural CBF operating in JEPA's latent space.

**Safety-filtered action:**
$$a^* = \arg\min_a \|a - a_{\text{VLA}}\|^2 \quad \text{s.t.} \quad h_\psi(P_\phi(a, s_t, z_t)) \geq -\alpha \cdot h_\psi(z_t)$$

### Feasibility
- **Compute:** Use frozen V-JEPA 2-AC (load pretrained weights), train only the CBF head (~small MLP)
- **Data:** LIBERO or DROID — add safety labels (e.g., collision/drop = unsafe)
- **Baselines:** SHIELD (CBF in state space), VLA-JEPA (no safety)
- **Novelty:** First CBF in JEPA latent space; first safety filter for JEPA-based VLAs

### Risk: Medium
The CBF needs to be effective in the latent space, which hasn't been proven.

### Paper Idea 1 (Deep Dive): CBF Mathematics for JEPA Latent Space

**Control Barrier Functions (CBFs)** provide formal safety guarantees. Here's the full mathematical framework:

**Definition — CBF in state space:**
Given a safe set $\mathcal{C} = \{x : h(x) \geq 0\}$, a function $h: \mathbb{R}^n \rightarrow \mathbb{R}$ is a CBF if:

$$\sup_{u \in U} \left[ \frac{\partial h}{\partial x} f(x, u) \right] \geq -\alpha(h(x))$$

where $\alpha$ is a class-$\mathcal{K}$ function (typically $\alpha(h) = \alpha_0 \cdot h$ for linear CBF).

**Translation to JEPA latent space:**
Replace state $x$ with JEPA latent state $z_t = E_{\bar\theta}(I_t)$, and dynamics $f(x,u)$ with the AC predictor:

$$h_\psi(z_{t+1}) \geq -\alpha \cdot h_\psi(z_t)$$
$$\text{where } z_{t+1} = P_\phi(a_t, s_t, z_t) \quad \text{(AC predictor)}$$

**Safety-filtered action (QP formulation):**
$$a^* = \arg\min_{a \in \mathbb{R}^7} \|a - a_{\text{VLA}}\|_2^2$$
$$\text{subject to: } h_\psi(P_\phi(a, s_t, z_t)) + \alpha \cdot h_\psi(z_t) \geq 0$$

This is a quadratic program (QP) — convex optimization, solvable in ~1ms.

**Training the neural CBF $h_\psi$:**
$$\mathcal{L}_{\text{CBF}} = \underbrace{\sum_{z \in \mathcal{D}_{\text{safe}}} \text{ReLU}(-h_\psi(z))}_{\text{safe states should have h>0}} + \underbrace{\sum_{z \in \mathcal{D}_{\text{unsafe}}} \text{ReLU}(h_\psi(z))}_{\text{unsafe states should have h<0}} + \lambda \underbrace{\mathcal{L}_{\text{descent}}}_{\text{CBF decrease condition}}$$

**Lyapunov stability connection:**
A CBF is essentially a Lyapunov function for the boundary of the safe set:
- If $h(z) > 0$, the system is safe (inside safe set)
- The CBF condition $h(z_{t+1}) \geq -\alpha h(z_t)$ ensures the system doesn't leave the safe set too quickly
- With $\alpha < 1$: even if $h(z_t)$ is small (near boundary), $h(z_{t+1})$ stays non-negative

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)

class LatentCBF(nn.Module):
    """Neural CBF operating in V-JEPA 2's latent space.
    
    h(z) > 0 → safe
    h(z) ≤ 0 → unsafe
    
    Trained on latent states labeled as safe/unsafe from demonstration data.
    """
    def __init__(self, latent_dim=1408, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),  # scalar output: h(z)
        )
    
    def forward(self, z):
        """z: [B, latent_dim] — mean-pooled JEPA latent state."""
        return self.net(z).squeeze(-1)  # [B] — positive = safe


def cbf_safety_filter(a_proposed, z_current, world_model, cbf, alpha=0.1, lr=0.01, steps=20):
    """Project proposed action to closest safe action using gradient descent on CBF constraint.
    
    Solves: min ||a - a_proposed||² s.t. h(world_model(a, z)) ≥ -α·h(z)
    
    In practice, you'd use a QP solver. Here we use projected gradient descent for simplicity.
    """
    a = a_proposed.clone().detach().requires_grad_(True)
    optimizer = torch.optim.Adam([a], lr=lr)
    
    h_current = cbf(z_current).detach()  # h(z_t)
    
    for _ in range(steps):
        # Predict next state
        z_next = world_model(z_current, a)  # simplified
        h_next = cbf(z_next)
        
        # CBF constraint: h(z_{t+1}) ≥ -α·h(z_t)
        constraint_violation = F.relu(-h_next - alpha * h_current)
        
        # Objective: stay close to proposed action + satisfy CBF
        loss = torch.sum((a - a_proposed.detach()) ** 2) + 100 * constraint_violation.sum()
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    return a.detach()


# Demo: 2D visualization of latent CBF
latent_dim = 2  # 2D for visualization
cbf = LatentCBF(latent_dim=latent_dim, hidden_dim=32)

# Create a grid of latent states
x = torch.linspace(-3, 3, 100)
y = torch.linspace(-3, 3, 100)
xx, yy = torch.meshgrid(x, y, indexing='ij')
grid = torch.stack([xx.flatten(), yy.flatten()], dim=-1)  # [10000, 2]

with torch.no_grad():
    h_values = cbf(grid).reshape(100, 100)

fig, ax = plt.subplots(figsize=(8, 6))
contour = ax.contourf(xx.numpy(), yy.numpy(), h_values.numpy(), levels=20, cmap='RdYlGn')
ax.contour(xx.numpy(), yy.numpy(), h_values.numpy(), levels=[0], colors='black', linewidths=2)
plt.colorbar(contour, label='h(z) — CBF value')
ax.set_xlabel('Latent dim 1')
ax.set_ylabel('Latent dim 2')
ax.set_title('Neural CBF in JEPA Latent Space\n'
             'Green: h(z)>0 (safe) | Red: h(z)<0 (unsafe) | Black line: boundary')
plt.tight_layout()
plt.show()

print("Paper Idea 1: Train this CBF on V-JEPA 2's 1408-dim latent states.")
print("Safety labels come from demonstrations (collision/drop/out-of-bounds = unsafe).")
print("At runtime: VLA proposes action → V-JEPA 2-AC predicts next state →")
print("CBF checks safety → QP corrects action if needed.")

In [ ]:
# CBF SAFETY FILTER: Complete demonstration with trajectory visualization

def demonstrate_cbf_safety_filter():
    """Show how a CBF safety filter corrects unsafe VLA actions."""
    
    torch.manual_seed(42)
    np.random.seed(42)
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    # Define a 2D safe region (for visualization)
    # Safe set: h(z) = 1 - (z[0]² + z[1]²)/R² > 0  (circle of radius R)
    R = 2.0
    
    def cbf_value(z):
        """h(z) = 1 - ||z||²/R² → safe if inside circle."""
        return 1.0 - (z[:, 0]**2 + z[:, 1]**2) / R**2
    
    # Plot 1: Safe set and CBF values
    ax = axes[0]
    x = np.linspace(-3, 3, 100)
    y = np.linspace(-3, 3, 100)
    X, Y = np.meshgrid(x, y)
    Z_cbf = 1.0 - (X**2 + Y**2) / R**2
    
    contour = ax.contourf(X, Y, Z_cbf, levels=20, cmap='RdYlGn', alpha=0.7)
    ax.contour(X, Y, Z_cbf, levels=[0], colors='black', linewidths=3)
    plt.colorbar(contour, ax=ax, label='h(z) — CBF value')
    
    theta_circle = np.linspace(0, 2*np.pi, 100)
    ax.plot(R*np.cos(theta_circle), R*np.sin(theta_circle), 'k-', linewidth=2)
    ax.set_xlabel('Latent dim 1')
    ax.set_ylabel('Latent dim 2')
    ax.set_title('Safe Set: h(z) > 0 (green)\nUnsafe: h(z) < 0 (red)')
    ax.set_aspect('equal')
    
    # Plot 2: VLA action vs CBF-corrected action
    ax = axes[1]
    
    # Start near the boundary
    z_current = torch.tensor([[1.5, 1.0]])
    h_current = cbf_value(z_current).item()
    
    # Simulate several VLA-proposed actions (some unsafe)
    n_actions = 8
    proposed_actions = torch.randn(n_actions, 2) * 0.5
    alpha_cbf = 0.3
    
    ax.contour(X, Y, Z_cbf, levels=[0], colors='black', linewidths=2)
    ax.plot(R*np.cos(theta_circle), R*np.sin(theta_circle), 'k-', linewidth=1, alpha=0.5)
    
    safe_count = 0
    corrected_count = 0
    
    for i in range(n_actions):
        a_proposed = proposed_actions[i]
        z_next_proposed = z_current[0] + a_proposed * 0.5  # simplified dynamics
        h_next = cbf_value(z_next_proposed.unsqueeze(0)).item()
        
        if h_next >= -alpha_cbf * h_current:
            # Safe — execute as-is
            ax.annotate('', xy=z_next_proposed.numpy(), xytext=z_current[0].numpy(),
                       arrowprops=dict(arrowstyle='->', color='green', linewidth=2))
            safe_count += 1
        else:
            # Unsafe — project to closest safe action
            # Simple projection: scale action to stay within safe boundary
            scale = 0.1
            while cbf_value((z_current[0] + a_proposed * scale * 0.5).unsqueeze(0)).item() < -alpha_cbf * h_current:
                scale *= 0.9
                if scale < 0.01:
                    break
            
            a_corrected = a_proposed * scale
            z_next_corrected = z_current[0] + a_corrected * 0.5
            
            # Draw proposed (red) and corrected (blue)
            ax.annotate('', xy=z_next_proposed.numpy(), xytext=z_current[0].numpy(),
                       arrowprops=dict(arrowstyle='->', color='red', linewidth=1.5, linestyle='--'))
            ax.annotate('', xy=z_next_corrected.numpy(), xytext=z_current[0].numpy(),
                       arrowprops=dict(arrowstyle='->', color='blue', linewidth=2))
            corrected_count += 1
    
    ax.plot(z_current[0, 0], z_current[0, 1], 'ko', markersize=10, zorder=10)
    ax.annotate('Current\nstate', xy=(z_current[0, 0].item(), z_current[0, 1].item()),
               xytext=(-30, 20), textcoords='offset points', fontsize=8)
    
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], color='green', linewidth=2, label=f'Safe actions ({safe_count})'),
        Line2D([0], [0], color='red', linewidth=1.5, linestyle='--', label=f'Proposed unsafe ({corrected_count})'),
        Line2D([0], [0], color='blue', linewidth=2, label=f'CBF-corrected ({corrected_count})'),
    ]
    ax.legend(handles=legend_elements, fontsize=8, loc='lower left')
    ax.set_xlabel('Latent dim 1')
    ax.set_ylabel('Latent dim 2')
    ax.set_title('CBF Safety Filter in Action\nRed=proposed, Blue=corrected')
    ax.set_xlim(-0.5, 3.5)
    ax.set_ylim(-1.5, 2.5)
    ax.set_aspect('equal')
    
    # Plot 3: h(z) evolution over a trajectory
    ax = axes[2]
    
    T_traj = 20
    z_traj = torch.tensor([[0.5, 0.5]])
    h_without_cbf = [cbf_value(z_traj).item()]
    h_with_cbf = [cbf_value(z_traj).item()]
    z_no_cbf = z_traj.clone()
    z_with_cbf = z_traj.clone()
    
    for t in range(T_traj):
        # Random action (pushing toward boundary)
        action = torch.tensor([[0.15, 0.08]]) + torch.randn(1, 2) * 0.05
        
        # Without CBF: just apply action
        z_no_cbf = z_no_cbf + action * 0.5
        h_without_cbf.append(cbf_value(z_no_cbf).item())
        
        # With CBF: check and correct
        z_next_candidate = z_with_cbf + action * 0.5
        h_next = cbf_value(z_next_candidate).item()
        h_curr = cbf_value(z_with_cbf).item()
        
        if h_next < -alpha_cbf * max(h_curr, 0.01):
            # Scale action
            scale = 0.5
            for _ in range(20):
                z_test = z_with_cbf + action * 0.5 * scale
                if cbf_value(z_test).item() >= -alpha_cbf * max(h_curr, 0.01):
                    break
                scale *= 0.8
            z_with_cbf = z_with_cbf + action * 0.5 * scale
        else:
            z_with_cbf = z_next_candidate
        
        h_with_cbf.append(cbf_value(z_with_cbf).item())
    
    steps = list(range(T_traj + 1))
    ax.plot(steps, h_without_cbf, 'r-o', linewidth=2, markersize=4, label='Without CBF')
    ax.plot(steps, h_with_cbf, 'b-o', linewidth=2, markersize=4, label='With CBF')
    ax.axhline(y=0, color='black', linewidth=2, linestyle='-', label='Safety boundary (h=0)')
    ax.fill_between(steps, -1, 0, color='red', alpha=0.1)
    ax.fill_between(steps, 0, 1.5, color='green', alpha=0.05)
    
    ax.set_xlabel('Time Step')
    ax.set_ylabel('h(z) — CBF Value')
    ax.set_title('CBF Keeps h(z) ≥ 0\n(system stays safe)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-0.5, 1.2)
    
    plt.tight_layout()
    plt.show()
    
    print("CBF Safety Filter for JEPA-VLA:")
    print("1. VLA proposes action a_VLA")
    print("2. World model predicts next state: ẑ_{t+1} = P(a_VLA, s_t, z_t)")
    print("3. CBF checks: h(ẑ_{t+1}) ≥ -α·h(z_t)?")
    print("4. If YES → execute a_VLA (safe)")
    print("5. If NO → solve QP: find closest safe action a* (~1ms)")
    print()
    print("Key advantage: safety in LATENT space is much faster than pixel space")
    print("because the CBF operates on a compact, semantically meaningful representation")

demonstrate_cbf_safety_filter()

## Paper Idea 2: Unified JEPA-VLA (Plug-in + World Model)

### Motivation
JEPA-VLA uses V-JEPA 2 features but has no world model.
VLA-JEPA has a world model but doesn't use V-JEPA 2 as a feature extractor for the VLM input.
**Combine both.**

### Architecture
```
                    ┌──────────────────────────────────────────────┐
                    │            Unified JEPA-VLA                   │
                    ├──────────────────────────────────────────────┤
                    │                                              │
Current frame ──►   │  Frozen V-JEPA 2 ──► Feature Extraction     │
                    │       │                    │                  │
                    │       │           ┌────────▼────────┐        │
                    │       │           │ VLM (Qwen/Llama)│        │
                    │       │           │ + V-JEPA 2 feats│        │
Language ──────►    │       │           │ (gated x-attn)  │        │
                    │       │           └────────┬────────┘        │
                    │       │                    │                  │
                    │       │              latent actions z_t       │
                    │       │                    │                  │
                    │       ▼           ┌────────▼────────┐        │
Future frames ──►   │  V-JEPA 2 enc ──►│  World Model    │        │
  (training only)   │  (targets)        │  (JEPA-style)   │        │
                    │                   └────────┬────────┘        │
                    │                            │                  │
                    │                    ┌───────▼───────┐         │
                    │                    │ Action Head   │         │
                    │                    │ (flow match)  │         │
                    │                    └───────┬───────┘         │
                    │                            │                  │
                    │                      robot actions            │
                    └──────────────────────────────────────────────┘
```

### Key Contributions
1. V-JEPA 2 features enhance VLM understanding (from JEPA-VLA)
2. World model in latent space enables planning (from VLA-JEPA)
3. Both use the SAME frozen V-JEPA 2 encoder — no extra compute for features

### Hypothesis
The world model loss provides an auxiliary gradient signal that further aligns
the VLM's internal representations with physical dynamics.

### Feasibility
- **Compute:** Qwen3-VL-2B is small; V-JEPA 2 is frozen; main cost is training the world model + action head
- **Data:** LIBERO (standard), DROID (for pretraining)
- **Baselines:** JEPA-VLA, VLA-JEPA, OpenVLA-OFT
- **Risk: LOW** — both components are proven individually

## Paper Idea 3: Amortized Planning with JEPA World Models

### Motivation
V-JEPA 2-AC plans via CEM (800 candidates, 5 iterations) → **16 seconds per action**.
This is unusable for real-time humanoid control (need <100ms).

### Approach: Distill CEM into a Policy
```
Training (offline):
  For each state z_t, goal z_g:
    1. Run CEM planner → optimal action a* (slow but accurate)
    2. Store (z_t, z_g, a*) in dataset
  Train policy π(z_t, z_g) → a* via behavior cloning

Inference (online):
  z_t, z_g → π → action in ~10ms (100x faster than CEM)
```

### Advanced: Plan-then-Refine
1. Amortized policy proposes initial action (fast, ~10ms)
2. 1-step CEM refinement using world model (~100ms)
3. Best of both: fast + accurate

### Feasibility
- **Compute:** CEM data generation can use V-JEPA 2-AC pretrained weights (no GPU needed for planning, just slow)
- **Data:** Generate planning dataset offline on CPU
- **Risk: LOW-MEDIUM** — behavior cloning from CEM is well-established

### Detailed Paper Structure for "Unified JEPA-VLA"

**Proposed ICRA paper outline (6 pages + 2 pages references):**

**I. Introduction (0.75 pages)**
- Problem: VLAs lack temporal understanding + forward modeling
- Recent advances: JEPA-VLA (features) and VLA-JEPA (world model)
- Our contribution: First unified approach combining both benefits
- Claim: synergistic improvement > sum of parts

**II. Related Work (0.75 pages)**
- VLAs: RT-2 → Octo → OpenVLA → pi0
- JEPA models: I-JEPA → V-JEPA → V-JEPA 2 → V-JEPA 2-AC
- JEPA + VLA: JEPA-VLA (plug-in), VLA-JEPA (end-to-end)
- Position: We combine both approaches for the first time

**III. Method (1.5 pages)**
- 3.1 Architecture overview (figure)
- 3.2 V-JEPA 2 feature injection via gated cross-attention (from JEPA-VLA)
- 3.3 Latent world model with leakage-free design (from VLA-JEPA)
- 3.4 Flow-matching action head (from pi0/VLA-JEPA)
- 3.5 Unified training objective: $\mathcal{L} = \mathcal{L}_{\text{FM}} + \beta\mathcal{L}_{\text{WM}} + \gamma\mathcal{L}_{\text{feat}}$

**IV. Experiments (2 pages)**
- 4.1 Setup: LIBERO benchmark, 4 task suites, evaluation protocol
- 4.2 Main results (Table 1): vs OpenVLA-OFT, JEPA-VLA, VLA-JEPA
- 4.3 Ablation study (Table 2): features only, WM only, both
- 4.4 Sample efficiency study (Figure): learning curves at 10-100% demos
- 4.5 Long-horizon analysis (Figure): per-task breakdown on LIBERO-Long
- 4.6 Qualitative analysis: when does each component help?

**V. Discussion & Conclusion (0.5 pages)**
- Key findings: features + WM is synergistic
- Limitations: compute, sim-only experiments
- Future work: safety (CBF), real robot, humanoid

**VI. References (2 pages)**

## Paper Idea 4: JEPA World Model for Sim-to-Real Transfer

### Motivation
V-JEPA 2 is pretrained on internet video → understands real-world physics.
Robot policies trained in simulation often fail in the real world (sim-to-real gap).

### Approach
Use V-JEPA 2's latent space as a **domain-invariant representation**:

```
Sim images → V-JEPA 2 encoder → latent state z_sim
Real images → V-JEPA 2 encoder → latent state z_real

Hypothesis: z_sim ≈ z_real for same physical configuration
because V-JEPA 2 focuses on semantics, not visual appearance.

→ Train VLA in simulation using z_sim
→ Deploy on real robot using z_real
→ No domain adaptation needed!
```

### Feasibility
- **Compute:** Only need simulation (free) + frozen V-JEPA 2
- **Data:** LIBERO in simulation → real Franka
- **Risk: MEDIUM** — depends on how well V-JEPA 2 representations generalize across domains

In [ ]:
# COMPUTE COST ANALYSIS: Detailed breakdown for ICRA paper experiments

def compute_cost_analysis():
    """Visualize compute requirements for the Unified JEPA-VLA paper."""
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    # Plot 1: Training cost breakdown
    ax = axes[0]
    
    components = ['V-JEPA 2\n(frozen)', 'VLM\n(Qwen3-VL-2B)', 'World Model\n(12-layer)', 
                  'Action Head\n(DiT-B)', 'Gated\nCross-Attn']
    gpu_hours = [0, 120, 40, 30, 20]  # estimated GPU hours on A100
    colors_bar = ['#95a5a6', '#3498db', '#2ecc71', '#e74c3c', '#f39c12']
    frozen = [True, False, False, False, False]
    
    bars = ax.bar(components, gpu_hours, color=colors_bar)
    for bar, is_frozen in zip(bars, frozen):
        if is_frozen:
            bar.set_hatch('///')
            bar.set_alpha(0.5)
    
    ax.set_ylabel('GPU Hours (A100)')
    ax.set_title('Training Cost per Component\n(hatched = frozen, no training)')
    ax.grid(True, alpha=0.3, axis='y')
    
    total = sum(gpu_hours)
    ax.text(0.95, 0.95, f'Total: ~{total} GPU-hrs\n≈ ${total * 2:.0f} on cloud\n≈ 1 day on 8×A100', 
            transform=ax.transAxes, ha='right', va='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='lightyellow'))
    
    # Plot 2: Memory usage
    ax = axes[1]
    
    models = ['V-JEPA 2\nViT-L\n(frozen)', 'V-JEPA 2\nViT-g\n(frozen)', 'Qwen3-VL\n2B', 
              'World Model\n12-layer', 'Action Head\nDiT-B']
    memory_gb = [1.2, 4.5, 4.0, 0.8, 0.5]
    fits_8gb = [True, False, True, True, True]
    
    bar_colors = ['#2ecc71' if fits else '#e74c3c' for fits in fits_8gb]
    ax.barh(models, memory_gb, color=bar_colors)
    ax.axvline(x=8.0, color='red', linewidth=2, linestyle='--', label='Your GPU: 8GB VRAM')
    ax.set_xlabel('GPU Memory (GB)')
    ax.set_title('Memory Requirements\nGreen = fits your 8GB, Red = needs cloud')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='x')
    
    # Plot 3: Experiment plan timeline
    ax = axes[2]
    ax.axis('off')
    
    experiment_plan = """
    EXPERIMENT PLAN (for ICRA paper)
    ═══════════════════════════════════

    Exp 1: Main Results (LIBERO suite)
    ──────────────────────────────────
    • 4 tasks × 5 seeds × 30K steps = 20 runs
    • ~200 GPU-hours on A100
    • Compare: OpenVLA-OFT, JEPA-VLA, VLA-JEPA, Unified (ours)

    Exp 2: Ablation Study
    ─────────────────────
    • Features only (no world model)
    • World model only (no JEPA features)
    • Both (full model)
    • ~60 GPU-hours

    Exp 3: Sample Efficiency
    ────────────────────────
    • 10%, 25%, 50%, 100% demos × 4 configs
    • ~80 GPU-hours

    Exp 4: Long-Horizon Analysis
    ────────────────────────────
    • LIBERO-Long (detailed per-task breakdown)
    • ~40 GPU-hours

    TOTAL: ~380 GPU-hours ≈ $760 cloud cost
    ═══════════════════════════════════
    """
    
    ax.text(0.05, 0.95, experiment_plan, fontfamily='monospace', fontsize=8.5,
            verticalalignment='top', transform=ax.transAxes,
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    ax.set_title('Experiment Plan')
    
    plt.tight_layout()
    plt.show()
    
    print("Budget-friendly strategy:")
    print("1. Prototype locally on CPU with small models (free)")
    print("2. Feature caching: run V-JEPA 2-L once, save features to disk")
    print("3. Training: Google Colab Pro ($10/mo) for small experiments")
    print("4. Final runs: Lambda Labs / RunPod ($2-3/hr per A100)")
    print("5. Total cloud cost estimate: $500-1000 for full paper")

compute_cost_analysis()

## Recommended Paper: Start with Idea 2 (Unified JEPA-VLA)

### Why This One?
1. **Lowest risk:** Both components proven individually
2. **Clear contribution:** First to combine plug-in features + world model
3. **Strong baselines:** JEPA-VLA and VLA-JEPA to compare against
4. **Manageable compute:** Frozen V-JEPA 2, train only world model + action head + VLM adapters
5. **Clear benchmarks:** LIBERO suite (standard for VLA papers)
6. **VLA-JEPA code is open source** — build on top of it

### Proposed Title
*"JEPA-Enhanced VLA with Latent World Modeling for Long-Horizon Robot Manipulation"*

### Abstract Draft

In [ ]:
abstract = """
PROPOSED ABSTRACT (Draft for ICRA 2026/2027)
==============================================

Vision-Language-Action (VLA) models have emerged as a scalable approach 
for robot manipulation, but their vision backbones lack physical 
understanding and their action prediction lacks forward modeling.
Recent work has shown that V-JEPA 2 video representations improve VLA 
perception (JEPA-VLA), and that JEPA-style world models enhance action 
prediction (VLA-JEPA). However, no prior work combines both benefits.

We propose [NAME], a unified framework that:
(1) injects frozen V-JEPA 2 video features into the VLM backbone via 
    gated cross-attention for physics-aware perception,
(2) uses a JEPA-style latent world model for future state prediction, and
(3) trains an action head via flow matching conditioned on both VLM 
    outputs and world model predictions.

On the LIBERO benchmark suite, [NAME] achieves XX.X% average success 
rate, improving over JEPA-VLA (+X.X%) and VLA-JEPA (+X.X%) while 
showing particular gains on long-horizon tasks (+X.X%). Ablation 
studies confirm that both the feature injection and world model 
contribute independently, and that their combination provides 
synergistic benefits. We also demonstrate improved sample efficiency:
[NAME] matches VLA-JEPA performance with 50% fewer robot demonstrations.
"""
print(abstract)

## Compute Planning (Your Hardware)

| Component | Your Hardware | Sufficient? | Alternative |
|-----------|---------------|-------------|-------------|
| V-JEPA 2 inference | RX 6700S 8GB | ❌ ViT-g is ~5GB+ | Use ViT-L (4GB) or CPU |
| VLM (Qwen3-VL-2B) | RX 6700S 8GB | ⚠️ Tight with ROCm | Use CPU or cloud burst |
| World model training | RX 6700S 8GB | ✅ Small model | Local |
| Action head training | RX 6700S 8GB | ✅ DiT-B fits | Local |
| LIBERO simulation | CPU | ✅ MuJoCo runs on CPU | Local |

### Recommended Setup
1. **Study & prototype:** Everything on CPU (small models, tiny batches)
2. **Feature extraction:** Run V-JEPA 2-large on CPU, cache features to disk
3. **Training:** Use Google Colab Pro ($10/mo) or Lambda Labs for GPU bursts
4. **Final experiments:** 8× A100 on cloud for ~2-3 days (LIBERO training is fast)

## Implementation Roadmap

### Phase 1: Reproduce Baselines (Weeks 1-4)
- [ ] Set up LIBERO environment locally
- [ ] Run VLA-JEPA code from `refs/VLA-JEPA/` on LIBERO
- [ ] Reproduce JEPA-VLA results (using their reported numbers if code unavailable)
- [ ] Cache V-JEPA 2 features for LIBERO dataset

### Phase 2: Build Unified Model (Weeks 5-8)
- [ ] Add gated cross-attention to VLA-JEPA's VLM (from JEPA-VLA paper)
- [ ] Feed V-JEPA 2 features to VLM alongside world model
- [ ] Verify training converges on LIBERO-Spatial (simplest)
- [ ] Run ablations: features only, world model only, both

### Phase 3: Full Experiments (Weeks 9-12)
- [ ] Train on all 4 LIBERO suites
- [ ] Long-horizon analysis (LIBERO-Long)
- [ ] Sample efficiency study (10%, 25%, 50%, 100% demos)
- [ ] Error analysis: when does each component help?

### Phase 4: Write Paper (Weeks 13-16)
- [ ] Introduction + related work
- [ ] Method section with architecture diagram
- [ ] Experiments + ablations
- [ ] Discussion: limitations, future work (safety/humanoid/real-time)

### Stretch Goals
- [ ] Real-robot validation (if lab access available)
- [ ] Add latent CBF safety filter (Paper Idea 1)
- [ ] Sim-to-real with frozen V-JEPA 2 features (Paper Idea 4)

## Key References for Your Paper

### Must-Cite Papers

| Paper | Year | Why Cite |
|-------|------|----------|
| V-JEPA 2 (Meta) | 2025 | Foundation model you build on |
| JEPA-VLA (Tsinghua) | 2026 | Feature injection approach |
| VLA-JEPA (USTC) | 2026 | World model approach |
| LeCun's vision paper | 2022 | JEPA conceptual foundation |
| OpenVLA | 2024 | VLA baseline |
| pi0 | 2024 | Flow matching for actions |
| LIBERO | 2023 | Benchmark suite |

### Should-Cite Papers

| Paper | Year | Why Cite |
|-------|------|----------|
| I-JEPA | 2023 | Original JEPA implementation |
| V-JEPA | 2024 | Video extension |
| Octo | 2024 | Generalist robot policy baseline |
| RT-2 | 2023 | Established VLA paradigm |
| SHIELD | 2025 | CBF for humanoids (if you add safety) |
| BarrierNet | 2023 | Neural CBFs (if you add safety) |

## Quick-Start: Getting the VLA-JEPA Code Running

The VLA-JEPA repo is cloned at `refs/VLA-JEPA/`. Here's how to start:

```bash
# 1. Create conda environment
conda create -n jepa-vla python=3.10 -y
conda activate jepa-vla

# 2. Install PyTorch (CPU for study, ROCm for AMD GPU)
pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu
# OR for ROCm (if available on Windows):
# pip install torch torchvision --index-url https://download.pytorch.org/whl/rocm6.2

# 3. Install dependencies
cd refs/VLA-JEPA
pip install -e .  # if setup.py exists

# 4. For V-JEPA 2 features:
pip install timm  # Vision Transformer library

# 5. For LIBERO:
pip install libero  # Robot learning benchmark
```

### Loading Pretrained V-JEPA 2 via PyTorch Hub

```python
import torch

# Load V-JEPA 2 ViT-Large (smallest, fits in 8GB)
model = torch.hub.load('facebookresearch/vjepa2', 'vjepa2_vit_large')
model.eval()

# Extract features from a video
video = torch.randn(1, 3, 16, 256, 256)  # [B, C, T, H, W]
with torch.no_grad():
    features = model(video)  # [1, 2048, 1024]
```

## Summary: Your Path to an ICRA Paper

```
NOW                    MONTH 1-2              MONTH 3-4              MONTH 5-6
────                   ────────               ────────               ────────
Study notebooks 01-04  Reproduce baselines    Build unified model    Write paper
Understand math        Run VLA-JEPA code      Full LIBERO exps       Submit to ICRA
Read key papers        Cache V-JEPA 2 feats   Ablation studies
Set up LIBERO env      Small-scale tests      Error analysis
```

### The One Thing to Start Tomorrow
1. Run `notebooks/01_JEPA_Fundamentals.ipynb` to verify your PyTorch works
2. Read the VLA-JEPA README at `refs/VLA-JEPA/README.md`
3. Try loading V-JEPA 2 ViT-Large via `torch.hub`

Good luck with ICRA! 🤖